# Phase 7 — Severity and Progression Feasibility Audit

This is a read-only audit of whether the available PlantVillage data and project metadata contain defensible ground-truth evidence for disease severity, grades/stages, disease-area measurements, temporal progression, and repeated observations of the same plant or leaf. It does not train a model.

Disease class names, visual symptoms, similar filenames, and image similarity are not severity labels, area measurements, timestamps, or plant/leaf identity.

## 1. Project and dataset path

Mount Google Drive in Colab first if needed. Set `TDF_DATASET_ROOT` to the existing PlantVillage `color` directory; the audit does not alter raw images. If the dataset is not mounted, the script reports that limitation and audits the tracked repository metadata only.

In [ ]:
from pathlib import Path
import os
import sys

repo_root = Path.cwd()
if not (repo_root / 'scripts' / 'audit_severity_progression.py').is_file():
    candidates = [parent for parent in repo_root.parents if (parent / 'scripts' / 'audit_severity_progression.py').is_file()]
    if candidates:
        repo_root = candidates[0]
if not (repo_root / 'scripts' / 'audit_severity_progression.py').is_file():
    raise FileNotFoundError('Run this notebook from the repository or clone the project first.')
os.chdir(repo_root)

if not os.environ.get('TDF_DATASET_ROOT'):
    os.environ['TDF_DATASET_ROOT'] = '/content/drive/MyDrive/UROP_Plant_Disease/datasets/PlantVillage/plantvillage dataset/color'
dataset_root = Path(os.environ['TDF_DATASET_ROOT']).expanduser()
print('Repository:', repo_root)
print('Dataset root:', dataset_root)
print('Dataset root available:', dataset_root.is_dir())

## 2. Environment verification

In [ ]:
import yaml
import pandas as pd

print('Python:', sys.version.split()[0])
print('PyYAML:', yaml.__version__)
print('pandas:', pd.__version__)
print('Environment ready for audit; no model training dependencies are invoked.')

## 3. Run the read-only audit script

The script reads structured metadata, manifests, and split tables. Candidate field names are discovery leads only; a populated field does not establish annotation provenance or validity.

In [ ]:
import subprocess

script_path = repo_root / 'scripts' / 'audit_severity_progression.py'
completed = subprocess.run(
    [sys.executable, str(script_path)],
    cwd=repo_root,
    env=os.environ.copy(),
    capture_output=True,
    text=True,
    check=False,
)
print(completed.stdout)
if completed.stderr:
    print(completed.stderr)
if completed.returncode != 0:
    raise RuntimeError(f'Audit script failed with exit code {completed.returncode}.')

## 4. Candidate metadata inventory

In [ ]:
import json

output_dir = repo_root / 'results' / 'phase7'
inventory_path = output_dir / 'severity_progression_candidate_inventory.csv'
report_path = output_dir / 'severity_progression_feasibility_report.yaml'
inventory = pd.read_csv(inventory_path)
with report_path.open('r', encoding='utf-8') as handle:
    report = yaml.safe_load(handle)

print('Dataset root status:', report['dataset_root_status'])
print('Dataset file scan:', report['dataset_file_scan'])
print('Metadata sources inventoried:', report['metadata_inventory_count'])
display(inventory if not inventory.empty else pd.DataFrame({'status': ['No structured candidate metadata sources found.']}))

## 5. Evidence-based target assessment

Only explicit severity/grade, disease-area, plant/leaf identity, and time fields are considered as possible metadata leads. The script does not infer labels from class names or estimate area from pixels.

In [ ]:
display(pd.DataFrame([
    {'Question': question.replace('_', ' ').capitalize(), 'Status': status}
    for question, status in report['questions'].items()
]))
print('Candidate field evidence:')
display(pd.DataFrame([
    {'Field group': group.replace('_', ' ').capitalize(), **evidence}
    for group, evidence in report['candidate_field_evidence'].items()
]) if report['candidate_field_evidence'] else pd.DataFrame({'Evidence': ['No populated candidate target fields were found in inspected metadata.']}))

## 6. Repeated-observation and progression checks

Repeated observations require an explicit plant/leaf identifier and distinct observation dates in the same metadata source. Duplicate images, filename similarity, and disease categories are not evidence of a longitudinal series.

In [ ]:
repeat_checks = []
for source in report['metadata_inventory']:
    check = source['repeated_observation_check']
    repeat_checks.append({
        'Source': source['source_path'],
        'Explicit identity fields': ', '.join(check['identity_fields']),
        'Explicit time fields': ', '.join(check['time_fields']),
        'Records with identity and date': check['records_with_identity_and_time'],
        'Identity groups with multiple dates': check['identity_groups_with_multiple_dates'],
        'Repeated observation evidence': check['repeated_observation_evidence'],
    })
display(pd.DataFrame(repeat_checks) if repeat_checks else pd.DataFrame({'Status': ['No structured sources available for an identity/date check.']}))
print('Candidate repeat-observation source files:', report['repeated_observation_candidate_sources'])

## 7. Phase 7 decision

A metadata field is not validated ground truth until its source, collection protocol, units/definitions, and linkage to the image are verified. This audit is a feasibility gate only; it does not fabricate severity labels, infer temporal progression, or train a severity/progression model.

In [ ]:
print(report['summary'])
display(pd.DataFrame([
    {'Capability': name.replace('_', ' ').capitalize(), 'Decision': status}
    for name, status in report['conclusions'].items()
]))
print('Manual review required:', report['manual_review_required'])
for limitation in report['evidence_and_limitations']:
    print('-', limitation)